In [1]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import  mean_absolute_error, r2_score
from sklearn import set_config
set_config(transform_output="pandas")

In [12]:
#Load dataset for Student Mathematics
df_math = pd.read_csv("student-mat.csv",sep=";")
# Adding a 'course' column to the Student Mathematics DataFrame
df_math['course'] = 'Math'
df_math.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,course
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,Math
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,Math
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,Math
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,Math
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,Math


In [13]:
#Load dataset for Student Portugese
df_por = pd.read_csv("student-por.csv",sep=";")
# Adding a 'course' column to the Student Portugese DataFrame
df_por['course'] = 'Portuguese'
df_por.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,course
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,4,0,11,11,Portuguese
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,2,9,11,11,Portuguese
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,6,12,13,12,Portuguese
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,0,14,14,14,Portuguese
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,0,11,13,13,Portuguese


In [19]:
# Display a random sample of 10 students from the combined dataset to check for both courses Math and Portuguese
print("Random sample of 10 students from both courses:")
print(combined_df.sample(10))

Random sample of 10 students from both courses:
    school sex  age address famsize Pstatus  Medu  Fedu      Mjob      Fjob  \
198     GP   F   17       U     GT3       T     4     4  services   teacher   
440     GP   F   15       U     LE3       A     4     3     other     other   
729     GP   M   18       U     GT3       T     2     1  services  services   
941     MS   F   17       U     GT3       T     4     4    health    health   
780     GP   F   18       U     GT3       T     2     2   at_home     other   
584     GP   M   17       U     LE3       T     4     3   teacher     other   
106     GP   F   15       U     GT3       T     2     2     other     other   
630     GP   F   17       U     GT3       T     1     1   at_home     other   
592     GP   F   17       U     LE3       T     3     3     other     other   
652     GP   M   17       U     GT3       T     4     4   teacher   teacher   

         reason guardian  traveltime  studytime  failures schoolsup famsup  \
198 

In [25]:
#Data Preparation and Feature Engineering

#Data Splitting: Dividing  datasets into training and testing sets
from sklearn.model_selection import train_test_split

# Define features (X) and target (y) for the math dataset
X_math = df_math.drop('G3', axis=1)
y_math = df_math['G3']

# Split the math data
X_math_train, X_math_test, y_math_train, y_math_test = train_test_split(X_math, y_math, test_size=0.2, random_state=42)

# Define features (X) and target (y) for the Portuguese dataset
X_por = df_por.drop('G3', axis=1)
y_por = df_por['G3']

# Split the Portuguese data
X_por_train, X_por_test, y_por_train, y_por_test = train_test_split(X_por, y_por, test_size=0.2, random_state=42)

# Define features (X) and target (y) for the combined dataset
X_combined = combined_df.drop('G3', axis=1)
y_combined = combined_df['G3']
#Split the combined data
X_combined_train, X_combined_test, y_combined_train, y_combined_test = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42)

In [28]:
#Preprocessing Pipelines: Defining how to transform raw data
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Separating numeric and categorical column names
numeric_features = X_math_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_math_train.select_dtypes(include=['object']).columns

# Creating the preprocessing pipelines for numeric (standarization) and categorical features (converting categorical feaures into numnerical format)
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

# Combining the transformers into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])


In [33]:
#Model Building and Training


from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

#List of models to be trained
models = [
    ('Linear Regression', LinearRegression()),
    ('Decision Tree', DecisionTreeRegressor(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42))
]

# Pipeline for math dataset with Linear Regression
math_lin_reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', LinearRegression())])

# Training and evaluate the math linear regression pipeline
math_lin_reg_pipeline.fit(X_math_train, y_math_train)
math_lin_reg_score = math_lin_reg_pipeline.score(X_math_test, y_math_test)
print(f"Math (Linear Regression) Test Score: {math_lin_reg_score}")

#Pipeline for math dataset with DecisionTreeRegressor
math_dec_tree_reg_pipeline =Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', DecisionTreeRegressor(random_state=42))])
# Training and evaluating the Decision Tree pipeline on the math data
math_dec_tree_reg_pipeline.fit(X_math_train, y_math_train)
math_dec_tree_reg_score = math_dec_tree_reg_pipeline.score(X_math_test, y_math_test)
print(f"Math (Decision Tree Regressor) Test Score: {math_dec_tree_reg_score}")

#Pipeline for math data with Radom Forrest Regressor
math_rand_for_reg_pipeline =Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', RandomForestRegressor(random_state=42))])

# Training and evaluating the Radom Forrest Regressor pipeline on the math data
math_rand_for_reg_pipeline.fit(X_math_train, y_math_train)
math_rand_for_reg_score = math_rand_for_reg_pipeline.score(X_math_test, y_math_test)
print(f"Math (Radom Forrest Regressor) Test Score: {math_rand_for_reg_score}")

Math (Linear Regression) Test Score: 0.7241341236974024
Math (Decision Tree Regressor) Test Score: 0.7339339855593412
Math (Radom Forrest Regressor) Test Score: 0.8079941088675648


1.   Math (Linear Regression) Test Score: 0.7241341236974024
2.   Math (Decision Tree Regressor) Test Score: 0.7339339855593412
3.   Math (Radom Forrest Regressor) Test Score: 0.8079941088675648

The R-square score ranges for 0 to 1. When the score is 1 indicates that the model perfectly predicts the final grade (G3). When the score is is 0, it mean the modeil is performing worse than a model that perdicts the mean.

Although all three models performed well, the Random Forrest Regressor was the best performing model. it showes an 80.8%  variance in G3 and was more effective at capturing the complex patterns in the data, than the other two models.





In [37]:
#Model Building and Training

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

#List of models to be trained
models = [
    ('Linear Regression', LinearRegression()),
    ('Decision Tree', DecisionTreeRegressor(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42))
]

# Pipeline for Portuguese dataset with Linear Regression
por_lin_reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', LinearRegression())])

# Training and evaluate the Portuguese linear regression pipeline
por_lin_reg_pipeline.fit(X_por_train, y_por_train)
por_lin_reg_score = por_lin_reg_pipeline.score(X_por_test, y_por_test)
print(f"Portuguese (Linear Regression) Test Score: {por_lin_reg_score}")

#Pipeline for Portuguese dataset with DecisionTreeRegressor
por_dec_tree_reg_pipeline =Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', DecisionTreeRegressor(random_state=42))])
# Training and evaluating the Decision Tree pipeline on the Portuguese data
por_dec_tree_reg_pipeline.fit(X_por_train, y_por_train)
por_dec_tree_reg_score = por_dec_tree_reg_pipeline.score(X_por_test, y_por_test)
print(f"Portuguese (Decision Tree Regressor) Test Score: {por_dec_tree_reg_score}")

#Pipeline for Portuguese data with Radom Forrest Regressor
por_rand_for_reg_pipeline =Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', RandomForestRegressor(random_state=42))])

# Training and evaluating the Radom Forrest Regressor pipeline on the Portuguese data
por_rand_for_reg_pipeline.fit(X_por_train, y_por_train)
por_rand_for_reg_score = por_rand_for_reg_pipeline.score(X_por_test, y_por_test)
print(f"Portuguese (Radom Forrest Regressor) Test Score: {por_rand_for_reg_score}")

Portuguese (Linear Regression) Test Score: 0.8486513286537314
Portuguese (Decision Tree Regressor) Test Score: 0.5961263076138928
Portuguese (Radom Forrest Regressor) Test Score: 0.8399277384044077


1. Portuguese (Linear Regression) Test Score: 0.8486513286537314
2. Portuguese (Decision Tree Regressor) Test Score: 0.5961263076138928
3. Portuguese (Radom Forrest Regressor) Test Score: 0.8399277384044077

For the R-square score, Lineat Regression Model shows approximately an 85% variance to G3 making it the best model compared to the other two tested.

In [39]:
from math import comb
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

#List of models to be trained
models = [
    ('Linear Regression', LinearRegression()),
    ('Decision Tree', DecisionTreeRegressor(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42))
]

# Pipeline for Combined dataset with Linear Regression
combined_lin_reg_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                        ('regressor', LinearRegression())])

# Training and evaluate the Combined linear regression pipeline
combined_lin_reg_pipeline.fit(X_combined_train, y_combined_train)
combined_lin_reg_score = combined_lin_reg_pipeline.score(X_combined_test, y_combined_test)
print(f"Combined (Linear Regression) Test Score: {combined_lin_reg_score}")

#Pipeline for Combined dataset with DecisionTreeRegressor
combined_dec_tree_reg_pipeline =Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', DecisionTreeRegressor(random_state=42))])
# Training and evaluating the Decision Tree pipeline on the Combined data
combined_dec_tree_reg_pipeline.fit(X_combined_train, y_combined_train)
combined_dec_tree_reg_score = combined_dec_tree_reg_pipeline.score(X_combined_test, y_combined_test)
print(f"Combined (Decision Tree Regressor) Test Score: {combined_dec_tree_reg_score}")

#Pipeline for Combined data with Radom Forrest Regressor
combined_rand_for_reg_pipeline =Pipeline(steps=[('preprocessor', preprocessor),
                                            ('regressor', RandomForestRegressor(random_state=42))])

# Training and evaluating the Radom Forrest Regressor pipeline on the Combined data
combined_rand_for_reg_pipeline.fit(X_combined_train, y_combined_train)
combined_rand_for_reg_score = combined_rand_for_reg_pipeline.score(X_combined_test, y_combined_test)
print(f"Combined (Radom Forrest Regressor) Test Score: {combined_rand_for_reg_score}")

Combined (Linear Regression) Test Score: 0.7996635665959496
Combined (Decision Tree Regressor) Test Score: 0.6738166030941268
Combined (Radom Forrest Regressor) Test Score: 0.830736379385913


1. Combined (Linear Regression) Test Score: 0.7996635665959496
2. Combined (Decision Tree Regressor) Test Score: 0.6738166030941268
3. Combined (Radom Forrest Regressor) Test Score: 0.830736379385913

Combined (Radom Forrest Regressor) Test Score: 0.830736379385913, shows a 83.07% varaince in G3 and is the best model in comparision to the other two models.

In [40]:
# Creating new features to address: Special attention should be given to how including prior grades (G1 and/or G2) changes model behavior.

# Define features (X) and target (y) for the math dataset *without* G1 and G2
X_math_no_g1g2 = df_math.drop(['G1', 'G2', 'G3'], axis=1)
y_math = df_math['G3']

# Define features (X) and target (y) for the Portuguese dataset *without* G1 and G2
X_por_no_g1g2 = df_por.drop(['G1', 'G2', 'G3'], axis=1)
y_por = df_por['G3']

# Define features (X) and target (y) for the combined dataset *without* G1 and G2
X_combined_no_g1g2 = combined_df.drop(['G1', 'G2', 'G3'], axis=1)
y_combined = combined_df['G3']


In [41]:
#Spliting new feature sets

from sklearn.model_selection import train_test_split

# Split the math data *without* G1 and G2
X_math_train_no_g1g2, X_math_test_no_g1g2, y_math_train, y_math_test = train_test_split(X_math_no_g1g2, y_math, test_size=0.2, random_state=42)

# Split the Portuguese data *without* G1 and G2
X_por_train_no_g1g2, X_por_test_no_g1g2, y_por_train, y_por_test = train_test_split(X_por_no_g1g2, y_por, test_size=0.2, random_state=42)

# Split the combined data *without* G1 and G2
X_combined_train_no_g1g2, X_combined_test_no_g1g2, y_combined_train, y_combined_test = train_test_split(X_combined_no_g1g2, y_combined, test_size=0.2, random_state=42)


In [44]:
#creating new datasets for traning loop

# Creatinng a dictionary of your datasets *without* G1 and G2
datasets_no_g1g2 = {
    'Math (no G1/G2)': (X_math_train_no_g1g2, X_math_test_no_g1g2, y_math_train, y_math_test),
    'Portuguese (no G1/G2)': (X_por_train_no_g1g2, X_por_test_no_g1g2, y_por_train, y_por_test),
    'Combined (no G1/G2)': (X_combined_train_no_g1g2, X_combined_test_no_g1g2, y_combined_train, y_combined_test)
}


In [49]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# --- Load and Combine Data ---
df_math = pd.read_csv("student-mat.csv", sep=';')
df_por = pd.read_csv("student-por.csv", sep=';')

df_math['course'] = 'Math'
df_por['course'] = 'Portuguese'

combined_df = pd.concat([df_math, df_por], ignore_index=True)

# Defining Features (X) and Target (y) for ALL scenarios
# Target is always G3
y_math = df_math['G3']
y_por = df_por['G3']
y_combined = combined_df['G3']

# Using all features (including G1, G2)
X_math = df_math.drop('G3', axis=1)
X_por = df_por.drop('G3', axis=1)
X_combined = combined_df.drop('G3', axis=1)

# Omitting G1 and G2
X_math_no_g1g2 = df_math.drop(['G1', 'G2', 'G3'], axis=1)
X_por_no_g1g2 = df_por.drop(['G1', 'G2', 'G3'], axis=1)
X_combined_no_g1g2 = combined_df.drop(['G1', 'G2', 'G3'], axis=1)

# Omitting only G2
X_math_no_g2 = df_math.drop(['G2', 'G3'], axis=1)
X_por_no_g2 = df_por.drop(['G2', 'G3'], axis=1)
X_combined_no_g2 = combined_df.drop(['G2', 'G3'], axis=1)


# Spliting Data for ALL scenarios
# The target (y) for each dataset remains constant, so we don't need new variables for them.
X_math_train, X_math_test, _, _ = train_test_split(X_math, y_math, test_size=0.2, random_state=42)
X_por_train, X_por_test, _, _ = train_test_split(X_por, y_por, test_size=0.2, random_state=42)
X_combined_train, X_combined_test, _, _ = train_test_split(X_combined, y_combined, test_size=0.2, random_state=42)

X_math_train_no_g1g2, X_math_test_no_g1g2, _, _ = train_test_split(X_math_no_g1g2, y_math, test_size=0.2, random_state=42)
X_por_train_no_g1g2, X_por_test_no_g1g2, _, _ = train_test_split(X_por_no_g1g2, y_por, test_size=0.2, random_state=42)
X_combined_train_no_g1g2, X_combined_test_no_g1g2, _, _ = train_test_split(X_combined_no_g1g2, y_combined, test_size=0.2, random_state=42)

X_math_train_no_g2, X_math_test_no_g2, _, _ = train_test_split(X_math_no_g2, y_math, test_size=0.2, random_state=42)
X_por_train_no_g2, X_por_test_no_g2, _, _ = train_test_split(X_por_no_g2, y_por, test_size=0.2, random_state=42)
X_combined_train_no_g2, X_combined_test_no_g2, _, _ = train_test_split(X_combined_no_g2, y_combined, test_size=0.2, random_state=42)


# Define Models and Automated Loop
models = [
    ('Linear Regression', LinearRegression()),
    ('Decision Tree', DecisionTreeRegressor(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42))
]

# Create a dictionary of all dataset splits for easy iteration
all_datasets = {
    'Math (with G1/G2)': (X_math_train, X_math_test, y_math_train, y_math_test),
    'Portuguese (with G1/G2)': (X_por_train, X_por_test, y_por_train, y_por_test),
    'Combined (with G1/G2)': (X_combined_train, X_combined_test, y_combined_train, y_combined_test),
    'Math (without G1/G2)': (X_math_train_no_g1g2, X_math_test_no_g1g2, y_math_train, y_math_test),
    'Portuguese (without G1/G2)': (X_por_train_no_g1g2, X_por_test_no_g1g2, y_por_train, y_por_test),
    'Combined (without G1/G2)': (X_combined_train_no_g1g2, X_combined_test_no_g1g2, y_combined_train, y_combined_test),
    'Math (without G2)': (X_math_train_no_g2, X_math_test_no_g2, y_math_train, y_math_test),
    'Portuguese (without G2)': (X_por_train_no_g2, X_por_test_no_g2, y_por_train, y_por_test),
    'Combined (without G2)': (X_combined_train_no_g2, X_combined_test_no_g2, y_combined_train, y_combined_test)
}

# Running the automated loop
results = {}
for dataset_name, (X_train, X_test, y_train, y_test) in all_datasets.items():
    print(f"--- Training models on {dataset_name} data ---")
    results[dataset_name] = {}

    current_numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
    current_categorical_features = X_train.select_dtypes(include=['object']).columns

    current_preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), current_numeric_features),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), current_categorical_features)
        ],
        remainder='passthrough'
    )

    for model_name, model in models:
        pipeline = Pipeline(steps=[('preprocessor', current_preprocessor),
                                   ('regressor', model)])

        pipeline.fit(X_train, y_train)
        score = pipeline.score(X_test, y_test)
        results[dataset_name][model_name] = score

        print(f"  - {model_name} Test Score: {score:.4f}")
    print("-" * 40)

# Printing all results in a summary table
results_df = pd.DataFrame(results)
print("\n--- Summary of All Model Scores ---")
print(results_df)



--- Training models on Math (with G1/G2) data ---
  - Linear Regression Test Score: 0.7241
  - Decision Tree Test Score: 0.7339
  - Random Forest Test Score: 0.8080
----------------------------------------
--- Training models on Portuguese (with G1/G2) data ---
  - Linear Regression Test Score: 0.8487
  - Decision Tree Test Score: 0.5961
  - Random Forest Test Score: 0.8399
----------------------------------------
--- Training models on Combined (with G1/G2) data ---
  - Linear Regression Test Score: 0.7997
  - Decision Tree Test Score: 0.6738
  - Random Forest Test Score: 0.8307
----------------------------------------
--- Training models on Math (without G1/G2) data ---
  - Linear Regression Test Score: 0.1415
  - Decision Tree Test Score: -0.0612
  - Random Forest Test Score: 0.2961
----------------------------------------
--- Training models on Portuguese (without G1/G2) data ---
  - Linear Regression Test Score: 0.1602
  - Decision Tree Test Score: -0.7417
  - Random Forest Test S

Questions and Answers:

a.	What is the difference in performance when predicting G3 by using all features, including G1 and G2, versus omitting G2, versus omitting both G1 and G2?

When removing both G1 and G2 the models performance drops dramatically. Observing the combined data set Linear Regression model R-square score is 0.799664 (With G1 & G2) and drops to  0.118546 (without G1 & G2). Even the best performing model Random Forrest only increases slighty.

Keepin all features have results in High r-square scores, with the benefit of pior grades (G1 and G2) acting as strong predictors. High degree of condifidence.

Only ommitting G2 shows perofmance reduction. A notable drop on the R-square score across the models. The scores are still slightly higher than when both G1 and G2 are omitted For instance, when Random Forest for Math goes from 0.8080 to 0.7323 without G2, but to 0.2961 without G1 and G2. This confirms that G1 still remains a very strong predictor.

-------------------------------------------------------------------------------
b.	Does the combined model perform better or worse than the separate models for each course, given these different feature sets?

The combined model’s performance is not consistent. It doesn't always beat or lose to the separate Math and Portuguese models. For Random Forest, it often performs well but for other models like Linear Regression, it gets a lower score than the individual course models.

-------------------------------------------------------------------------------
c.	Which regression algorithm among the three works best under each version (with/without prior grades, separate/combined)?

The Random Forest Regressor generally works better. It constantly achieved the highest R-squared scores the majority of scenarios. The only exception being Portuguese data with G1/G2, where Linear Regression was slightly better.

------------------------------------------------------------------------------
d.	What trade-offs emerge in terms of prediction accuracy vs interpretability vs early‐actionability (for example, excluding G2 because it may not yet be available)?

Accuracy vs. Early Action:

If we need to know a student's final grade early in the semester, the prediction won't be very accurate. The most accurate prediction comes after we have some of their actual grades to use.

Accuracy vs. Interpretability:

-The Random Forest model is like a highly skilled. It makes the best final grade predictions, but it's hard to explain exactly how it reached that conclusion.
-The Linear Regression model is like a simple recipe. It's easier to understand because you can see how each component affects the outcome, but the final result isn't as good as the highly skilled model.

-------------------------------------------------------------------------------
e.	What are the practical implications of choosing a version with prior grades for educators or policy makers, especially if they want to intervene early.

We can see that, using models with prior grades gives high prediction accuracy.However, it is only useful for reactive intervention later in the semester. For proactive early intervention, educators must use the much less accurate models without prior grades, meaning they must accept a margin of error (false positives). This strategy would be best for bringing attention to students who need closer monitoring from the start.